In [5]:
import numpy as np

from weather.config import (
    Experiment,
    WeatherFixedParams,
    WeatherGridParams,
    MLPFixedParams,
    MLPGridParams,
    FitFixedParams,
    FitGridParams,
)

from weather.search import Search

from mlp.utils import (
    plot_loss,
    regression_report,
    classification_report_binary,
    plot_roc_auc,
    plot_accuracy,
    accuracy_within_tolerance,
)


In [6]:
SEED = 42
np.random.seed(SEED)


In [7]:
# =========================================================
# WIND (>=6 m/s) BINARY CLASSIFICATION
# =========================================================
exp_wind_encoding = Experiment(
    name="wind6_binary_encoding",

    # =========================
    # WEATHER
    # =========================
    weather_fixed=WeatherFixedParams(
        target="wind_speed",
        target_mode="binary",
        target_threshold=6.0,
        data_dir="../data",
        skip_day=True,
        # normalization="global",
        encode_wind_direction=True,
    ),

    weather_grid=WeatherGridParams(
        window_aggregation=["flatten", "aggregate"],
        window_size=3,
        normalization=["standardize", "minmax", "l2", "none"],
        input_variables=[
            ("wind_speed",),
            ("wind_speed", "wind_direction"),
            ("wind_speed", "wind_direction", "pressure"),
            ("wind_speed", "wind_direction", "pressure", "humidity"),
            ("wind_speed", "wind_direction", "pressure", "humidity", "temperature"),
        ],
        aggregations=[{
            "temperature": ("mean", "min", "max"),
            "humidity": ("mean", "min", "max"),
            "pressure": ("mean", "min", "max"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        },
        {
            "temperature": ("mean", "min", "max", "trend"),
            "humidity": ("mean", "min", "max", "trend"),
            "pressure": ("mean", "min", "max", "trend"),
            "wind_speed": ("mean", "max"),
            "wind_direction": ("mean",),
        }
        ],
        cities=("Vancouver",),
        # cities=[
        #     ("Vancouver",),
        #     ("Vancouver", "Seattle", "Portland"),
        #     ("Beersheba", "Tel Aviv District", "Eilat", "Haifa", "Nahariyya", "Jerusalem")
        # ]
    ),

    # =========================
    # MLP
    # =========================
    mlp_fixed=MLPFixedParams(
        task="binary",
        beta = 0.9,
        beta2 = 0.999,
        eps = 1e-8,
        adaptive_lr=True,
        lr_decay=0.99,
    ),

    mlp_grid=MLPGridParams(
        hidden_layers=(32, 64),
        loss="binary_cross_entropy",
        activation="gelu",
        learning_rate=0.01,
        seed=SEED,
        use_bias=True,
        optimizer="momentum",
    ),

    # =========================
    # FIT
    # =========================
    fit_fixed=FitFixedParams(
        verbose=True,
        log_every=None,
        use_tqdm=True,
        one_hot_if_needed=True,
        early_stopping=True,
        patience=50,
        min_delta=0.0001,
    ),

    fit_grid=FitGridParams(
        epochs=400,
        batch_size="auto",
        shuffle=False,
        val_split=0.1,
    ),
)


In [8]:
search = Search()
results = search.run(exp_wind_encoding)


Starting experiment: wind6_binary_encoding

Building dataset
  → TRAIN split


Vancouver | windows: 100%|██████████| 1518/1518 [00:02<00:00, 651.68it/s]


  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 719.15it/s]



Configuration run 1/80:
WEATHER (variable):
  - window_aggregation: flatten
  - input_variables: ('wind_speed',)
  - aggregations: {'temperature': ('mean', 'min', 'max'), 'humidity': ('mean', 'min', 'max'), 'pressure': ('mean', 'min', 'max'), 'wind_speed': ('mean', 'max'), 'wind_direction': ('mean',)}
  - normalization: standardize

Training model


...

  → TEST split


Vancouver | windows: 100%|██████████| 361/361 [00:00<00:00, 587.33it/s]



Configuration run 80/80:
WEATHER (variable):
  - window_aggregation: aggregate
  - input_variables: ('wind_speed', 'wind_direction', 'pressure', 'humidity', 'temperature')
  - aggregations: {'temperature': ('mean', 'min', 'max', 'trend'), 'humidity': ('mean', 'min', 'max', 'trend'), 'pressure': ('mean', 'min', 'max', 'trend'), 'wind_speed': ('mean', 'max'), 'wind_direction': ('mean',)}
  - normalization: none

Training model


Training:  99%|█████████▉| 395/400 [00:27<00:00, 14.14it/s, acc=0.6713, loss=0.6077, lr=0.000188756]

Early stopping at epoch 396, best val_loss=0.686337, train_acc=0.6713, val_acc=0.5592 after 50 epochs without improvement.
Training finished in 27.93 seconds

Experiment finished | total runs = 80



In [11]:
from IPython.core.display import HTML

for run in results:
    model = run["model"]
    y_test = run["y_test"]
    y_pred = run["y_pred"]
    y_proba = run["y_proba"]

    history = run["history"]
    accuracy_history = run["accuracy_history"]
    config_log = run.get("config_log", {})

    print("\n\n" + "=" * 80)
    print(f"EXPERIMENT: {run['experiment']}")
    print(f"TASK: {model.task.upper()}")
    print("=" * 80)

    # ========= CONFIGURATION (VARIABLE PARAMS) =========
    if config_log:
        print("CONFIGURATION (variable params):")
        for section, params in config_log.items():
            if not params:
                continue
            print(f"  {section.upper()}:")
            for k, v in params.items():
                print(f"    - {k}: {v}")
        print("-" * 80)

    # ===================== METRICS =====================
    if model.task == "binary":
        metrics = classification_report_binary(
            y_true=y_test,
            y_score=y_proba,
            threshold=0.5,
        )

        auc_val = metrics["auc"]
        if auc_val >= 0.54:
            auc_color = "#2e7d32"   # dark green
        elif auc_val >= 0.53:
            auc_color = "#558b2f"   # olive green
        elif auc_val >= 0.52:
            auc_color = "#f9a825"   # amber
        elif auc_val >= 0.51:
            auc_color = "#ef6c00"   # orange
        else:
            auc_color = "#c62828"   # red

        print("=== TEST METRICS (BINARY CLASSIFICATION) ===")
        print(f"Accuracy : {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall   : {metrics['recall']:.4f}")
        print(f"Auc      : {metrics['auc']:.4f}")
        # display(HTML(
        #     f"""
        #     <div style="
        #         font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
        #         font-size: 13px;
        #         color: {auc_color};
        #         padding-left: 12px;
        #         margin: 4px 0;
        #     ">
        #         <b>AUC</b>: {auc_val:.4f}
        #     </div>
        #     """
        # ))

    else:
        metrics = regression_report(
            y_true=y_test,
            y_pred=y_pred,
        )

        acc_2 = accuracy_within_tolerance(y_test, y_pred, tol=2.0)
        acc_25 = accuracy_within_tolerance(y_test, y_pred, tol=2.5)

        if acc_2 >= 0.62:
            color = "#2e7d32"   # dark green
        elif acc_2 >= 0.61:
            color = "#558b2f"   # olive green
        elif acc_2 >= 0.60:
            color = "#f9a825"   # amber
        elif acc_2 >= 0.58:
            color = "#ef6c00"   # orange
        else:
            color = "#c62828"   # red

        print("=== TEST METRICS (REGRESSION) ===")
        print(f"MAE              : {metrics['mae']:.4f}")
        print(f"MSE              : {metrics['mse']:.4f}")
        print(f"RMSE             : {metrics['rmse']:.4f}")
        # display(HTML(
        #     f"""
        #     <div style="
        #         font-family: 'JetBrains Mono', 'Consolas', 'Menlo', monospace;
        #         font-size: 13px;
        #         color: {color};
        #         padding-left: 12px;
        #         margin: 4px 0;
        #     ">
        #         <b>Accuracy |err|≤2°C</b>: {acc_2:.4f}
        #     </div>
        #     """
        # ))
        print(f"Accuracy |err|≤2°C   : {acc_2:.4f}")
        print(f"Accuracy |err|≤2.5°C : {acc_25:.4f}")

    # --- plots ---
    # plot_loss(
    #     history,
    #     title="Train Loss Evolution",
    # )
    #
    # if model.task != "regression":
    #     if accuracy_history and not all(np.isnan(accuracy_history)):
    #         plot_accuracy(
    #             accuracy_history,
    #             title="Accuracy evolution",
    #         )
    #
    # # ROC only for binary classification
    # if model.task == "binary":
    #     plot_roc_auc(
    #         y_true=y_test,
    #         y_score=y_proba,
    #         title="ROC Curve",
    #     )




EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_aggregation: flatten
    - input_variables: ('wind_speed',)
    - aggregations: {'temperature': ('mean', 'min', 'max'), 'humidity': ('mean', 'min', 'max'), 'pressure': ('mean', 'min', 'max'), 'wind_speed': ('mean', 'max'), 'wind_direction': ('mean',)}
    - normalization: standardize
--------------------------------------------------------------------------------
=== TEST METRICS (BINARY CLASSIFICATION) ===
Accuracy : 0.5244
Precision: 0.6023
Recall   : 0.5521
Auc      : 0.5367


EXPERIMENT: wind6_binary_encoding
TASK: BINARY
CONFIGURATION (variable params):
  WEATHER:
    - window_aggregation: flatten
    - input_variables: ('wind_speed',)
    - aggregations: {'temperature': ('mean', 'min', 'max'), 'humidity': ('mean', 'min', 'max'), 'pressure': ('mean', 'min', 'max'), 'wind_speed': ('mean', 'max'), 'wind_direction': ('mean',)}
    - normalization: minmax
-------------------------